# 01 — Thu thập và khám phá dữ liệu

## 1. Bối cảnh dữ liệu

- **Chủ đề:** anime và người dùng MyAnimeList (MAL), nơi người dùng tự lập danh sách anime và có thể chấm điểm 1–10.
- **Đơn vị quan sát:** có 3 bảng, mỗi dòng lần lượt là một anime, một user, và một cặp user–anime.
- **Nguồn:** [MyAnimeList Dataset trên Kaggle](https://www.kaggle.com/datasets/azathoth42/myanimelist), tác giả Matěj Račinský, 2018, doi: 10.34740/KAGGLE/DSV/45582.
- **Giấy phép:** Kaggle ghi "Database: Open Database, Contents: Database Contents". Tác giả cho phép dùng không cần xin phép thêm hay trả phí, và yêu cầu trích dẫn như trên khi dùng trong sản phẩm nghiên cứu hoặc thuyết trình.
- **Cách thu thập:** crawl dữ liệu công khai từ MyAnimeList.net. MAL định danh bằng username nên không duyệt tuần tự được; tác giả lấy username từ các chủ đề forum watching challenge 2015–2018 và từ khoảng 40 nghìn câu lạc bộ đầu tiên (trong khoảng 80 nghìn).
- **Hạn chế đã biết:**
  - User đến từ forum và câu lạc bộ, nên mẫu có thể lệch về nhóm tích cực, không đại diện cho toàn bộ người dùng MAL.
  - Dữ liệu dừng ở khoảng năm 2018.
  - Mỗi cặp user–anime chỉ giữ trạng thái và lần cập nhật cuối, không có lịch sử thay đổi.
- **Phiên bản dùng:** 3 file raw (`AnimeList.csv`, `UserList.csv`, `UserAnimeList.csv`). Bản filtered và cleaned do tác giả xử lý, chỉ dùng để đối chiếu.
- **Lý do chọn dữ liệu:** **TODO (nhóm viết)**

**Điều kiện của đề:** dữ liệu công khai, có thật, truy được nguồn, từ khoảng 3,000 dòng và 10 cột trở lên, không phải dữ liệu tổng hợp.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Chạy được cả trên Kaggle lẫn trong repo (khi đặt dữ liệu ở data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Không tìm thấy dữ liệu: cần chạy trên Kaggle hoặc trong repo.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Số file CSV:", len(paths))

## 2. Đọc dữ liệu và đối chiếu với mô tả

**Ý nghĩa một dòng của `UserAnimeList.csv`:** một user và một anime trong danh sách của user đó. Bước này chỉ đọc 4 cột cần cho đối chiếu vì file nặng khoảng 5 GB.

| Cột | Ý nghĩa | Loại dữ liệu |
|---|---|---|
| `username` | Định danh user | Khóa |
| `anime_id` | Định danh anime | Khóa |
| `my_score` | Điểm user chấm, 1–10 theo mô tả | Thứ tự |
| `my_status` | Kệ mà user xếp anime vào | Danh nghĩa: mã số chỉ là nhãn, tính trung bình là vô nghĩa |

Mã `my_status` theo mô tả: 1 watching, 2 completed, 3 on hold, 4 dropped, 6 plan to watch; các giá trị khác không rõ.

Trước khi viết `assert`, ta in số thực tế cạnh số trong mô tả, vì một số giá trị có thể lệch vì lý do hợp lệ.

In [ ]:
anime_raw = pd.read_csv(paths["AnimeList.csv"])
users_raw = pd.read_csv(paths["UserList.csv"])
lists_raw = pd.read_csv(
    paths["UserAnimeList.csv"],
    usecols=["username", "anime_id", "my_score", "my_status"],
    dtype={"username": "category", "anime_id": "int32",
           "my_score": "int32", "my_status": "int32"},
)

# So sánh với mô tả trước, chưa assert
checks = {
    "records":       (len(lists_raw), 80_076_112),
    "records rated": (int((lists_raw.my_score > 0).sum()), 46_358_322),
    "users (lists)": (lists_raw.username.nunique(), 302_675),
    "users (table)": (users_raw.username.nunique(), 302_675),
    "anime (table)": (anime_raw.anime_id.nunique(), 14_478),
    "anime (lists)": (lists_raw.anime_id.nunique(), 14_478),
}
for name, (actual, expected) in checks.items():
    print(f"{name:15s} thực tế={actual:>12,} mô tả={expected:>12,} lệch={actual - expected:+,}")

print("anime_id duy nhất:", anime_raw.anime_id.is_unique)
print("title duy nhất   :", anime_raw.title.is_unique)
print(lists_raw.my_status.value_counts().sort_index())
print(lists_raw.my_score.value_counts().sort_index())

### 2.1 Kết quả đối chiếu với mô tả

**Khớp:** 80,076,112 record, 46,358,322 record có rating, 14,478 anime ở cả hai bảng.

**Khác với mô tả:**

| Mục | Mô tả | Thực tế | Lệch | Giải thích | Trạng thái |
|---|---|---|---|---|---|
| Username trong `UserAnimeList` | 302,675 | 283,044 | −19,631 | Có thể là danh sách trống hoặc crawl thiếu | Chưa kiểm chứng |
| Username trong `UserList` | 302,675 | 302,674 | −1 | Preview của Kaggle đếm được 302,675, nên có thể pandas đọc một tên thành `NaN` | Chưa kiểm chứng |
| Giá trị `my_status` | 1, 2, 3, 4, 6 | Có thêm 0, 5, 33, 55 | 537 dòng | Mô tả ghi các giá trị khác không rõ | Cần quyết định khi làm sạch |

**Đọc output của hai cột chính:**

| `my_status` | Nghĩa | Số dòng | Tỷ lệ |
|---|---|---|---|
| 2 | completed | 50,445,526 | 63.0% |
| 6 | plan to watch | 19,392,104 | 24.2% |
| 1 | watching | 4,057,788 | 5.1% |
| 4 | dropped | 3,194,353 | 4.0% |
| 3 | on hold | 2,985,804 | 3.7% |
| 0, 5, 33, 55 | không rõ | 537 | dưới 0.001% |

- **`my_score = 0` nghĩa là chưa chấm.** Mô tả ghi thang điểm 1–10, và số dòng có điểm từ 1 trở lên khớp đúng 46,358,322 record có rating. Có 33,717,790 dòng điểm 0 (42.1%), nên phải đổi thành thiếu trước khi tính trung bình.
- **Điểm 0 không có nghĩa là đã xem mà không chấm,** vì plan to watch thường chưa xem. Cần tách theo `my_status`, xem ô tiếp theo.
- **Trên các dòng có chấm, điểm trung bình là khoảng 7.49 và lệch về phía cao:** 76.3% từ 7 trở lên, 5.6% từ 4 trở xuống, điểm phổ biến nhất là 8.
- **`title` không duy nhất,** nên dùng `anime_id` làm khóa.

In [ ]:
# 1) Viết thành assert những gì đã đối chiếu khớp
assert len(lists_raw) == 80_076_112
assert (lists_raw.my_score > 0).sum() == 46_358_322
assert lists_raw.my_score.between(0, 10).all()
assert anime_raw.anime_id.nunique() == 14_478
assert lists_raw.anime_id.nunique() == 14_478
assert anime_raw.anime_id.is_unique

# 2) Mỗi trạng thái có bao nhiêu dòng chưa chấm, và điểm trung bình bao nhiêu?
print(lists_raw.groupby("my_status").my_score.agg(
    rows="size",
    unrated=lambda s: (s == 0).sum(),
    pct_unrated=lambda s: round((s == 0).mean() * 100, 1),
    mean_rated=lambda s: s[s > 0].mean(),
))

# 3) Các dòng có status ngoài mô tả
known_status = [1, 2, 3, 4, 6]
odd_status = lists_raw[~lists_raw.my_status.isin(known_status)]
print("\nstatus lạ:", len(odd_status), "| số user:", odd_status.username.nunique())
print(odd_status.head(8))

# 4) Username bị thiếu: đọc lại dạng chuỗi thô để xem tên thật là gì
username_text = pd.read_csv(paths["UserList.csv"], usecols=["username"],
                            dtype=str, keep_default_na=False)["username"]
print("\nusers_raw số dòng:", len(users_raw),
      "| username NaN:", users_raw.username.isna().sum(),
      "| tên thô tương ứng:", username_text[users_raw.username.isna()].tolist(),
      "| username trùng:", users_raw.username.dropna().duplicated().sum())
print("lists username NaN:", lists_raw.username.isna().sum())

# 5) User không có dòng nào trong lists: danh sách trống hay crawl thiếu?
list_users = set(lists_raw.username.cat.categories)
table_users = set(users_raw.username.dropna())
print("có trong lists, không có trong table:", len(list_users - table_users))
count_cols = ["user_watching", "user_completed", "user_onhold",
              "user_dropped", "user_plantowatch"]
users_no_list = users_raw[~users_raw.username.isin(list_users)]
declared = users_no_list[count_cols].sum(axis=1)
print("user không có list:", len(users_no_list),
      "| khai báo 0 anime:", (declared == 0).sum(),
      "| khai báo > 0:", (declared > 0).sum())
print(declared[declared > 0].describe())

# 6) Khóa của lists, và hai tập anime có thật sự trùng nhau không?
print("\ntrùng (username, anime_id):",
      lists_raw.duplicated(["username", "anime_id"]).sum())
print("anime_id lệch giữa hai bảng:",
      len(set(lists_raw.anime_id) ^ set(anime_raw.anime_id)))

# 7) Anime trùng tên
same_title = anime_raw[anime_raw.title.duplicated(keep=False)]
print("\nanime trùng title:", len(same_title))
print(same_title.sort_values("title")[
    ["anime_id", "title", "type", "aired_string"]].head(10))

### 2.2 Kiểm tra các điểm lệch

**TODO (điền sau khi chạy ô trên):** ghi số thật và kết luận cho từng mục.

| Mục trong ô trên | Câu hỏi cần trả lời | Kết luận |
|---|---|---|
| 2 | Người drop có ít chấm hơn người xem xong không, và điểm thấp hơn bao nhiêu? Plan to watch có dòng nào được chấm không? | TODO |
| 3 | Status lạ tập trung ở vài user hay rải rác? Loại hay giữ? | TODO |
| 4 | Username bị đọc thành `NaN` là tên gì? | TODO |
| 5 | 19,631 user có danh sách trống hay crawl thiếu? Nhóm "khai báo > 0" càng lớn thì càng nghiêng về crawl thiếu. | TODO |
| 6 | Mỗi cặp user–anime có duy nhất không? Hai tập `anime_id` có trùng nhau không? | TODO |
| 7 | Có bao nhiêu anime trùng tên, và chúng có thật sự là các anime khác nhau không? | TODO |

## 3. Data quality exploration

**TODO:** Document duplicates / empty rows, numerical distributions / implausible values / outliers, categorical frequencies / inconsistent labels, missingness patterns and relationships relevant to candidate questions. Explain why each issue matters, not merely print outputs.

In [ ]:
# Examples; choose only meaningful checks once data is loaded.
# display(missing_overview(raw_df))
# print("Duplicate rows:", raw_df.duplicated().sum())
# display(raw_df.describe(include="all").T)


## 4. Cleaning decisions, then save processed dataset

**TODO:** For each cleaning step, explain the rule and why it is justified. Use `report_row_change(step, before_df, after_df, reason)` for filtering, dropping and merging. Preserve original raw data. Do not assume all missing values or outliers should be removed.

In [ ]:
# Example pattern only; replace with a justified operation.
# before_df = raw_df.copy()
# processed_df = before_df.copy()  # TODO: actual cleaning rule
# report_row_change("Step name", before_df, processed_df, "Explain the operation")
# processed_df.to_csv(ROOT / "data/processed/cleaned.csv", index=False)


## 5. Exploration summary

Write **3–5 evidence-based observations**, serious quality problems, a preprocessing plan and possible directions for research questions. Keep observation separate from interpretation. (Team must write this section themselves.)